# Pipeline de profiling taxonomique (Sylph + GlobDB)

## Ce notebook fait quoi ?
Ce notebook lance **Sylph** sur des paires de reads FASTQ, genere des profils taxonomiques par echantillon, puis assemble une matrice d'abondance pour analyse comparative, avec la base **GlobDB**.

## Entrees attendues
- Dossier des reads: `data/fastq`
- Fichiers apparies: `sample_R1.fastq.gz` et `sample_R2.fastq.gz`
- Base Sylph GlobDB (IFB core cluster):
  - `SYLPH_DB = /data/sylph/databases/sylph/globdb/globdb_r226_sylph_c200.syldb`

## Etapes executees
1. Configuration des chemins (`DATA_DIR`, `RESULT_DIR`, `SYLPH_DB`) et creation des dossiers de sortie.
2. Detection automatique des echantillons a partir des `R1`.
3. Execution de `sylph profile` pour chaque paire (`R1`, `R2`).
4. Lecture/aggregation des TSV produits en table unique d'abondances relatives.
5. Export de la matrice combinee (`combined_sylph_abundance.tsv`).
6. Filtrage des taxons faiblement abondants et clustering hierarchique (clustermap).

## Sorties principales
- Resultats par echantillon dans `results_profiling/sylph/`
- Matrice combinee: `results_profiling/combined_sylph_abundance.tsv`

## Points de vigilance
- Verifier l'accessibilite du path IFB GlobDB Sylph.
- Verifier la presence des fichiers `R2` pour chaque echantillon.
- Les commandes shell dans notebook (`!module load ...`) dependent de votre environnement cluster.

In [ ]:
import os
import glob
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

In [ ]:
# --- CONFIGURATION DES PATHS ---
DATA_DIR = "data/fastq"  # Dossier contenant vos .fastq.gz
RESULT_DIR = "results_profiling"
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
# Base de donnees GlobDB pour Sylph (path IFB core cluster)
SYLPH_DB = "/data/sylph/databases/sylph/globdb/globdb_r226_sylph_c200.syldb"

In [ ]:
# Liste des échantillons (on suppose un format sample_R1.fastq.gz)
fastq_r1s = sorted(glob.glob(f"{DATA_DIR}/*_R1.fastq.gz"))
samples = [os.path.basename(f).split('_R1')[0] for f in fastq_r1s]

In [ ]:
# Dossier spécifique pour Sylph
sylph_out = f"{RESULT_DIR}/sylph"
os.makedirs(sylph_out, exist_ok=True)

In [ ]:
for r1 in fastq_r1s:
    sample = os.path.basename(r1).split('_R1')[0]
    r2 = r1.replace("_R1", "_R2")
    output = f"{sylph_out}/{sample}.tsv"
    
    print(f"🚀 Processing {sample} with Sylph...")
    
    # On charge le module et on lance l'outil dans la même ligne
    !module load sylph && sylph profile {SYLPH_DB} {r1} {r2} -t 8 -o {output}

In [ ]:
# Agrégation des résultats Sylph
sylph_tables = []
for s in samples:
    path = f"{sylph_out}/{s}.tsv"
    if os.path.exists(path):
        df = pd.read_csv(path, sep='\t')
        # On garde le nom du taxon et la proportion (abondance relative)
        df = df[['tax_name', 'proportion']].rename(columns={'proportion': s})
        sylph_tables.append(df.set_index('tax_name'))

In [ ]:
df_sylph_final = pd.concat(sylph_tables, axis=1).fillna(0)
df_sylph_final.to_csv(f"{RESULT_DIR}/combined_sylph_abundance.tsv", sep='\t')

In [ ]:
# 📊 3. Analyse Comparative & Clustering

In [ ]:
# Choix de la table à analyser
data_to_plot = df_sylph_final.copy()

# Filtrage : On ne garde que les taxons qui atteignent au moins 1% quelque part
data_filtered = data_to_plot[data_to_plot.max(axis=1) >= 0.01]

print(f"Nombre de taxons après filtrage (>1%) : {len(data_filtered)}")

In [ ]:
# --- CLUSTERING HIERARCHIQUE ---
# Calcul de la distance (Euclidienne) et du lien (Ward)
Z = linkage(pdist(data_filtered.T), method='ward')

# Affichage du Clustermap
g = sns.clustermap(data_filtered, 
                   method='ward', 
                   cmap="YlGnBu", 
                   figsize=(12, 10),
                   xticklabels=True, 
                   yticklabels=True,
                   cbar_kws={'label': 'Abondance Relative'})

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
plt.suptitle(f"Heatmap de clustering hiérarchique (Top {len(data_filtered)} taxons)", fontsize=16)
plt.show()